# PCU-OBJECTIVE-ALIGNMENT-001 — Final objective-only diagnostic

Final engineering diagnostic. It compares L7/K64 CE against the same L7/K64 Cell set trained with exact 16-way context-oracle-v2 candidate ranking.

If the historical locality-width artifacts are unavailable because the prior Kaggle working directory was ephemeral, this notebook reruns **only K64 CE** as a paired control. It never reconstructs or republishes K16/K32 as if the historical sweep still existed.

Formal seeds are never executed.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
LAYER_BASELINE = REPO / 'artifacts/research/pcu-layer-placement-001/engineering/26090501-layer-only'
LOCALITY = REPO / 'artifacts/research/pcu-locality-width-001/engineering/26090501-l7-width'
OUT = REPO / 'artifacts/research/pcu-objective-alignment-001/engineering/26090501-l7-k64-ranking'
PAIRED = OUT / 'PAIRED_CE_K64.json'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'transformers': transformers.__version__,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
assert run(['git', 'hash-object', SEED_REGISTRY], capture=True) == '71a3015a7d54e795538b3aa6750860f0b9168cb3'
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU objective-alignment test/compile gate: PASS')


In [ ]:
# Resolve the CE K64 baseline. Prefer the complete historical locality sweep.
full_required = ['RUN_IDENTITY.json', 'DESIGN.json', 'DECISION.json', 'WIDTH_016.json', 'WIDTH_032.json', 'WIDTH_064.json']
full_locality = all((LOCALITY / name).is_file() for name in full_required)
remote_path = 'artifacts/research/pcu-locality-width-001/engineering/26090501-l7-width/DECISION.json'
remote_has = subprocess.run(['git', 'show', f'origin/{BRANCH}:{remote_path}'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0

if full_locality:
    locality_decision = json.loads((LOCALITY / 'DECISION.json').read_text())
    assert locality_decision['status'] == 'LOCALITY_WIDTH_IMPROVES_BUT_DOES_NOT_RESCUE'
    width64 = json.loads((LOCALITY / 'WIDTH_064.json').read_text())
    assert width64['identity']['selected_k'] == 64
    assert len(width64['allocation']['selected']) == 64
    assert abs(width64['direct_accuracy'] - 0.265625) < 1e-12
    if not remote_has:
        print('Publishing complete historical locality-width prerequisite...')
        run([sys.executable, 'scripts/research/publish_pcu_locality_width_001.py', '--branch', BRANCH])
        run(['git', 'fetch', 'origin'])
    baseline_mode = 'historical_locality_sweep'
else:
    print('Historical locality artifacts unavailable in this Kaggle session; recovering only the exact L7/K64 CE paired control.')
    OUT.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable, 'scripts/research/recover_pcu_objective_k64_control.py',
        '--seed', '26090501',
        '--device', 'cuda:0',
        '--baseline', LAYER_BASELINE,
        '--locality-root', LOCALITY,
        '--out', PAIRED,
    ])
    paired = json.loads(PAIRED.read_text())
    assert paired['recovery_mode'] == 'paired_ce_k64_only'
    assert paired['historical_sweep_not_reconstructed'] is True
    assert paired['selected_k'] == 64
    assert len(paired['selected_cells']) == 64
    assert abs(paired['direct_accuracy'] - 0.265625) < 1e-12
    width64 = json.loads((LOCALITY / 'WIDTH_064.json').read_text())
    assert width64['allocation']['selected'] == paired['selected_cells']
    baseline_mode = 'paired_ce_k64_recovery'

assert formal_states() == expected
print(json.dumps({
    'baseline_mode': baseline_mode,
    'k64_direct_accuracy': width64['direct_accuracy'],
    'k64_gradient_mass': width64['allocation']['gradient_mass_at_k'],
    'historical_sweep_reconstructed': False if baseline_mode == 'paired_ce_k64_recovery' else None,
}, indent=2))


In [ ]:
# PAIRED_CE_K64.json is allowed to exist before the ranking arm. Final artifacts are not.
final_names = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json']
already_final = [name for name in final_names if (OUT / name).is_file()]
assert not already_final, f'Final objective output already exists; inspect before rerun: {already_final}'
run([
    sys.executable, 'scripts/research/run_pcu_objective_alignment_001.py',
    '--seed', '26090501',
    '--device', 'cuda:0',
    '--baseline', LOCALITY,
    '--out', OUT,
])


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
assert decision['valid_run'] is True
assert decision['formal_execution_not_started'] is True
assert decision['selected_cells_exact_baseline_match'] is True
assert result['selected_cells'] == width64['allocation']['selected']
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'baseline_mode': baseline_mode,
    'ce_baseline_direct_accuracy': decision['ce_baseline_direct_accuracy'],
    'ranking_train_accuracy': decision['ranking_train_accuracy'],
    'ranking_eval_accuracy': decision['ranking_eval_accuracy'],
    'direct_accuracy': decision['direct_accuracy'],
    'training_final_rank_loss': result['training']['final_loss'],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_objective_alignment_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'baseline_mode': baseline_mode, 'formal_seed_states': formal_states()}, indent=2))
